# 06 - Simulación del Seguro Indexado

Este notebook evalúa el desempeño técnico-financiero del seguro construido a partir del índice de riesgo climático.

**Entradas principales**

- `data/outputs/indice_riesgo_climatico_2007-2024.csv`
- `data/model/parametros_seguro.csv`
- `data/model/precios_cafe_anual_2007-2027.csv`

**Salidas principales**

- `data/outputs/resultado_anual_seguro.csv`
- `data/outputs/resultado_cluster_seguro.csv`
- `data/outputs/resultado_riesgo_seguro.csv`
- `data/outputs/simulacion_montecarlo.csv`
- `data/outputs/resumen_portafolio_seguro.csv`

El objetivo es evaluar primas, indemnizaciones, resultado técnico, loss ratio y riesgo financiero del portafolio.


## 1. Librerías

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 2. Rutas y parámetros

In [ ]:
# ============================================================
# RUTAS ROBUSTAS DEL PROYECTO
# ============================================================

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PATH_MODEL = PROJECT_ROOT / "data" / "model"
PATH_OUTPUTS = PROJECT_ROOT / "data" / "outputs"

PATH_OUTPUTS.mkdir(parents=True, exist_ok=True)

# Entradas
INPUT_INDICE = PATH_OUTPUTS / "indice_riesgo_climatico_2007-2024.csv"
INPUT_PARAMETROS = PATH_MODEL / "parametros_seguro.csv"
INPUT_PRECIOS = PATH_MODEL / "precios_cafe_anual_2007-2027.csv"

# Salidas
OUTPUT_RESUMEN_PORTAFOLIO = PATH_OUTPUTS / "resumen_portafolio_seguro.csv"
OUTPUT_RESULTADO_ANUAL = PATH_OUTPUTS / "resultado_anual_seguro.csv"
OUTPUT_RESULTADO_CLUSTER = PATH_OUTPUTS / "resultado_cluster_seguro.csv"
OUTPUT_RESULTADO_RIESGO = PATH_OUTPUTS / "resultado_riesgo_seguro.csv"
OUTPUT_MONTECARLO = PATH_OUTPUTS / "simulacion_montecarlo.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_INDICE:", INPUT_INDICE)
print("INPUT_PARAMETROS:", INPUT_PARAMETROS)
print("INPUT_PRECIOS:", INPUT_PRECIOS)


In [ ]:
# ============================================================
# PARÁMETROS DE SIMULACIÓN
# ============================================================

N_SIM = 5000
N_POLIZAS = 100
RANDOM_STATE = 42

# Si no se encuentra columna de área sembrada, se simula por hectárea.
AREA_DEFAULT_HA = 1.0


## 3. Carga de insumos

In [ ]:
df = pd.read_csv(INPUT_INDICE)
parametros = pd.read_csv(INPUT_PARAMETROS)
precios = pd.read_csv(INPUT_PRECIOS)

params = parametros.iloc[0].to_dict()

print("Filas índice:", len(df))
print("Columnas índice:", df.columns.tolist()[:10], "...")
print("Parámetros cargados:")
print(parametros.T)


In [ ]:
# Validaciones mínimas
cols_necesarias = [
    "municipio", "anio", "cluster",
    "Rendimiento (t/ha)", "riesgo",
    "payout", "evento_perdida_global"
]

faltantes = [c for c in cols_necesarias if c not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas necesarias en el índice: {faltantes}")

# Nombre esperado de precio anual
if "precio_referencia_USD_ton" in precios.columns:
    col_precio = "precio_referencia_USD_ton"
elif "PM30_prom_USD_ton" in precios.columns:
    col_precio = "PM30_prom_USD_ton"
else:
    raise ValueError("No se encontró columna de precio anual compatible.")

print("Columna de precio usada:", col_precio)


## 4. Preparación económica del portafolio

El índice climático produce un `payout` relativo entre 0 y 1.

Para expresarlo económicamente se calcula un monto asegurado aproximado:

```text
monto_asegurado = rendimiento_promedio_municipal × precio_café × área
```

Este supuesto permite evaluar el seguro en términos monetarios relativos.  
En la API final, el usuario podrá ingresar su área cultivada y el sistema recalculará el monto asegurado individual.


In [ ]:
# Unir precio anual
precios_aux = precios[["anio", col_precio]].copy()
precios_aux = precios_aux.rename(columns={col_precio: "precio_USD_ton"})

df = df.merge(precios_aux, on="anio", how="left")

if df["precio_USD_ton"].isna().sum() > 0:
    raise ValueError("Existen registros sin precio anual asignado.")

# Área
if "Área sembrada (ha)" in df.columns:
    df["area_ha_simulada"] = df["Área sembrada (ha)"].fillna(AREA_DEFAULT_HA)
else:
    df["area_ha_simulada"] = AREA_DEFAULT_HA

# Rendimiento esperado base por municipio
df["rendimiento_esperado_mun"] = (
    df.groupby("municipio")["Rendimiento (t/ha)"].transform("mean")
)

# Monto asegurado
df["monto_asegurado"] = (
    df["rendimiento_esperado_mun"] *
    df["precio_USD_ton"] *
    df["area_ha_simulada"]
)

# Tasas desde parámetros
prima_pura_rate = float(params["prima_pura"])
prima_comercial_rate = float(params["prima_comercial"])

df["prima_pura"] = prima_pura_rate * df["monto_asegurado"]
df["prima_comercial"] = prima_comercial_rate * df["monto_asegurado"]
df["indemnizacion"] = df["payout"] * df["monto_asegurado"]

df["resultado_puro"] = df["prima_pura"] - df["indemnizacion"]
df["resultado_comercial"] = df["prima_comercial"] - df["indemnizacion"]

df[[
    "municipio", "anio", "cluster", "riesgo",
    "Rendimiento (t/ha)", "precio_USD_ton",
    "area_ha_simulada", "monto_asegurado",
    "prima_comercial", "indemnizacion",
    "resultado_comercial"
]].head()


## 5. Resumen general del portafolio

In [ ]:
resumen_portafolio = pd.DataFrame([{
    "n_observaciones": len(df),
    "n_municipios": df["municipio"].nunique(),
    "anio_min": df["anio"].min(),
    "anio_max": df["anio"].max(),
    "prima_pura_total": df["prima_pura"].sum(),
    "prima_comercial_total": df["prima_comercial"].sum(),
    "indemnizacion_total": df["indemnizacion"].sum(),
    "resultado_puro_total": df["resultado_puro"].sum(),
    "resultado_comercial_total": df["resultado_comercial"].sum(),
    "loss_ratio_puro": df["indemnizacion"].sum() / df["prima_pura"].sum(),
    "loss_ratio_comercial": df["indemnizacion"].sum() / df["prima_comercial"].sum(),
    "payout_promedio": df["payout"].mean(),
    "tasa_evento_perdida": df["evento_perdida_global"].mean()
}])

resumen_portafolio


## 6. Resultado anual del seguro

In [ ]:
resultado_anual = (
    df
    .groupby("anio")
    .agg(
        n=("anio", "size"),
        prima_total=("prima_comercial", "sum"),
        indemnizacion_total=("indemnizacion", "sum"),
        resultado_total=("resultado_comercial", "sum"),
        payout_promedio=("payout", "mean"),
        tasa_evento_perdida=("evento_perdida_global", "mean"),
        monto_asegurado_total=("monto_asegurado", "sum")
    )
    .reset_index()
)

resultado_anual["loss_ratio"] = (
    resultado_anual["indemnizacion_total"] /
    resultado_anual["prima_total"]
)

resultado_anual


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(resultado_anual["anio"], resultado_anual["prima_total"], marker="o", label="Primas")
plt.plot(resultado_anual["anio"], resultado_anual["indemnizacion_total"], marker="o", label="Indemnizaciones")
plt.xlabel("Año")
plt.ylabel("USD")
plt.title("Primas vs indemnizaciones por año")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(resultado_anual["anio"], resultado_anual["resultado_total"])
plt.axhline(0, linestyle="--")
plt.xlabel("Año")
plt.ylabel("USD")
plt.title("Resultado técnico anual del seguro")
plt.grid(axis="y")
plt.show()


## 7. Resultado por clúster

In [ ]:
resultado_cluster = (
    df
    .groupby("cluster")
    .agg(
        n=("cluster", "size"),
        prima_total=("prima_comercial", "sum"),
        indemnizacion_total=("indemnizacion", "sum"),
        resultado_total=("resultado_comercial", "sum"),
        payout_promedio=("payout", "mean"),
        tasa_evento_perdida=("evento_perdida_global", "mean"),
        monto_asegurado_total=("monto_asegurado", "sum")
    )
    .reset_index()
)

resultado_cluster["loss_ratio"] = (
    resultado_cluster["indemnizacion_total"] /
    resultado_cluster["prima_total"]
)

resultado_cluster


## 8. Resultado por nivel de riesgo

In [ ]:
resultado_riesgo = (
    df
    .groupby("riesgo")
    .agg(
        n=("riesgo", "size"),
        prima_total=("prima_comercial", "sum"),
        indemnizacion_total=("indemnizacion", "sum"),
        resultado_total=("resultado_comercial", "sum"),
        payout_promedio=("payout", "mean"),
        tasa_evento_perdida=("evento_perdida_global", "mean"),
        rendimiento_promedio=("Rendimiento (t/ha)", "mean"),
        monto_asegurado_total=("monto_asegurado", "sum")
    )
    .reset_index()
)

resultado_riesgo["loss_ratio"] = (
    resultado_riesgo["indemnizacion_total"] /
    resultado_riesgo["prima_total"]
)

resultado_riesgo


## 9. Simulación Monte Carlo

Se realiza remuestreo histórico del portafolio para evaluar estabilidad financiera.

Cada simulación toma `N_POLIZAS` observaciones con reemplazo.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
simulaciones = []

for i in range(N_SIM):

    muestra = df.sample(
        n=N_POLIZAS,
        replace=True,
        random_state=int(rng.integers(0, 1_000_000))
    )

    prima_total = muestra["prima_comercial"].sum()
    indemnizacion_total = muestra["indemnizacion"].sum()
    resultado_total = prima_total - indemnizacion_total
    loss_ratio = indemnizacion_total / prima_total if prima_total > 0 else np.nan

    simulaciones.append({
        "sim": i,
        "prima_total": prima_total,
        "indemnizacion_total": indemnizacion_total,
        "resultado_total": resultado_total,
        "loss_ratio": loss_ratio
    })

df_sim = pd.DataFrame(simulaciones)

df_sim.describe()


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df_sim["resultado_total"], bins=40)
plt.axvline(0, linestyle="--")
plt.xlabel("Resultado técnico")
plt.ylabel("Frecuencia")
plt.title("Monte Carlo - Resultado técnico")
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df_sim["loss_ratio"], bins=40)
plt.axvline(1, linestyle="--")
plt.xlabel("Loss ratio")
plt.ylabel("Frecuencia")
plt.title("Monte Carlo - Loss ratio")
plt.grid(True)
plt.show()


## 10. Métricas de riesgo financiero

In [ ]:
metricas_financieras = pd.DataFrame([{
    "probabilidad_perdida_aseguradora": (df_sim["resultado_total"] < 0).mean(),
    "loss_ratio_promedio": df_sim["loss_ratio"].mean(),
    "loss_ratio_mediano": df_sim["loss_ratio"].median(),
    "loss_ratio_p95": df_sim["loss_ratio"].quantile(0.95),
    "resultado_promedio": df_sim["resultado_total"].mean(),
    "resultado_p05": df_sim["resultado_total"].quantile(0.05),
    "resultado_p95": df_sim["resultado_total"].quantile(0.95)
}])

metricas_financieras


## 11. Guardar salidas

In [ ]:
resumen_portafolio.to_csv(OUTPUT_RESUMEN_PORTAFOLIO, index=False)
resultado_anual.to_csv(OUTPUT_RESULTADO_ANUAL, index=False)
resultado_cluster.to_csv(OUTPUT_RESULTADO_CLUSTER, index=False)
resultado_riesgo.to_csv(OUTPUT_RESULTADO_RIESGO, index=False)
df_sim.to_csv(OUTPUT_MONTECARLO, index=False)

# También guardar el dataset enriquecido de simulación para auditoría
OUTPUT_BASE_SIM = PATH_OUTPUTS / "base_simulacion_seguro.csv"
df.to_csv(OUTPUT_BASE_SIM, index=False)

print("Archivos guardados:")
print(OUTPUT_RESUMEN_PORTAFOLIO)
print(OUTPUT_RESULTADO_ANUAL)
print(OUTPUT_RESULTADO_CLUSTER)
print(OUTPUT_RESULTADO_RIESGO)
print(OUTPUT_MONTECARLO)
print(OUTPUT_BASE_SIM)


## 12. Siguiente paso

Este notebook deja lista la evaluación financiera del producto.

El siguiente paso es construir:

```text
07_modelo_seguro_completo.ipynb
```

Ese notebook debe tomar como entradas:

- `features_intra_anuales_2025-2027.csv`
- `clusters_municipales.csv`
- `coeficientes_indice_por_cluster.csv`
- `variables_indice_final.csv`
- `parametros_seguro.csv`
- `precios_cafe_anual_2007-2027.csv`

y devolver para un usuario:

- producción esperada
- rendimiento esperado
- índice de riesgo
- nivel de riesgo
- prima
- monto asegurado
- indemnización estimada
